In [ ]:
import pandas as pd
import looker_sdk
from io import StringIO

In [ ]:
# Load the Looker Python SDK tools.
import looker_sdk

# Path to the `looker.ini` file we setup.


CREDENTIALS_PATH = "utils\\looker.ini"  # <--- Change


# Initializing the Looker SDK.


sdk = looker_sdk.init40(CREDENTIALS_PATH)

# Retrieve my user information from Looker.
my_user = sdk.me()

# Display my username.
my_user["display_name"]

In [ ]:
#############
# Parameters
#############
RESULT_FORMAT = "csv"

#############
# Send Query
#############

# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "agencies",
    "fields": [
        "agencies.name",
        "primary_sites.coc",
        "programs.name",
        "programs.availability_end_date",
        "programs.availability_start_date",
        "programs.status",
        "programs.id",
        "programs.project_type_code",
        # "program_inventory.id",
        "program_inventory.bed_inventory",
        # "program_inventory.bed_ch_veteran",
        # "program_inventory.bed_ch_youth",
        # "program_inventory.bed_non_dedicated",
        # "program_inventory.bed_ch_other",
        # "program_inventory.bed_other_veteran",
        # "program_inventory.bed_other_youth",
        # "program_inventory.bed_youth_veteran",
        "program_inventory_pdm.bed_inventory_ap",
        # "program_inventory_pdm.total_bed_inventory_ap",
    ],
    "filters": {},
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df_1 = pd.read_csv(StringIO(result))

In [ ]:
df_1

In [ ]:
df_1

In [ ]:
df_1 = df_1[
    (
        df_1["Programs Project Type Code"].isin(
            ["Transitional Housing", "Safe Haven", "Emergency Shelter – Entry Exit"]
        )
    )
    & (df_1["Programs Program Active"] == "Active")
].copy()

In [ ]:
#############
# Parameters
#############
RESULT_FORMAT = "csv"

#############
# Send Query
#############

# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": [
        "programs.id",
        "enrollments.count",
        "household_makeup.household_type",
        "enrollments.total_days_in_project_during_reporting_period",
    ],
    "filters": {"enrollments.date_filter": "1 month"},
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df_2 = pd.read_csv(StringIO(result))

In [ ]:
df_2[df_2["Programs Program ID"] == 113]

In [ ]:
df_1["Programs Program ID"] = df_1["Programs Program ID"].astype("Int64").copy()

In [ ]:
df_1.info()

In [ ]:
df_1["Programs Operating End Date"].fillna("9999-12-31", inplace=True)

In [ ]:
df_1 = df_1.groupby(
    [
        "Agencies Agency Name",
        "Programs CoC",
        "Programs Name",
        "Programs Operating End Date",
        "Programs Operating Start Date",
        "Programs Program Active",
        "Programs Program ID",
        "Programs Project Type Code",
    ],
    as_index=False,
)["Bed and Unit Inventory Total Bed Inventory for Analysis Period"].sum()

In [ ]:
df_2.info()

In [ ]:
merged_df = pd.merge(df_1, df_2, how="inner", on="Programs Program ID")

In [ ]:
merged_df = merged_df[
    merged_df["Programs Project Type Code"].isin(
        ["Transitional Housing", "Safe Haven", "Emergency Shelter – Entry Exit"]
    )
]

In [ ]:
import numpy as np

In [ ]:
merged_df["Utilization"] = merged_df[
    "Enrollments Total Days in Project During the Reporting Period"
] / merged_df["Bed and Unit Inventory Total Bed Inventory for Analysis Period"].replace(
    0, np.nan
)

In [ ]:
merged_df

In [ ]:
from ds_sql_connections import GsAutomationReliefAssistanceConnection
import os
import json

main_directory_path = str(os.getcwd())
# Path to the config file
config_path = main_directory_path + "\\utils\\config_partners_in_care.json"
# creating the config object to pull the authentication credentials later in various calls.
with open(config_path, "r", encoding="utf-8") as file:
    config_object = json.load(file)

# sql connection credentials
sql_username = config_object["Josh"]["username"]
sql_password = config_object["Josh"]["password"]
conn = GsAutomationReliefAssistanceConnection(sql_username, sql_password, auth=True)
conn.insert_with_progress(
    df=merged_df,
    schema="dbo",
    tbl_nm="partners_in_care_utilization_original",
    dtypes={},
)